### Clean up the raw LLM input
There are some paragraphs with '\n' in the sentence, which makes the .tsv have multiple line for each abstract/paragraph, leading to a wrong number of rows and incorrect assignment for a abs/papa to PMCID/PMID. Therefore, a clean up is reuiqred to make sure each row is a unique record of a asbtract / paragraph.

In [17]:
import os

input_dir = "/mnt/isilon/wang_lab/shared/pubmed_central_additional/batch_BERT_filtered_input/"
output_dir = "/mnt/isilon/wang_lab/shared/pubmed_central_additional/batch_BERT_filtered_input_clean"

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# Loop through all TSV files in the input directory
counter=1
for filename in os.listdir(input_dir):
    print(counter,end=' ')
    if filename.endswith(".tsv"):
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)

        with open(input_path, "r", encoding="utf-8") as infile, open(output_path, "w", encoding="utf-8") as outfile:
            buffer = ""
            for line in infile:
                line = line.rstrip("\n")
                if line.startswith("PMC"):
                    if buffer:
                        outfile.write(buffer + "\n")
                    buffer = line
                else:
                    buffer += " " + line.strip()
            if buffer:
                outfile.write(buffer + "\n")

    counter+=1
print("All files cleaned and saved to:", output_dir)


1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250 251 252 253 254 255 256 257 258 259 260 261 262 263 264 265 266 267 268 269 270 271 272 273 274 275 276 277 

In [19]:
import os
import re

input_dir = "/mnt/isilon/wang_lab/shared/pubmed/pubmed_BERT_filtered_input/"
output_dir = "/mnt/isilon/wang_lab/shared/pubmed/pubmed_BERT_filtered_input_clean"

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# Loop through all TSV files in the input directory
counter=1
for filename in os.listdir(input_dir):
    print(counter,end=' ')
    if filename.endswith(".xml"):
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)

        with open(input_path, "r", encoding="utf-8") as infile, open(output_path, "w", encoding="utf-8") as outfile:
            header = infile.readline()  # keep header
            outfile.write(header)

            buffer = ""
            for line in infile:
                line = line.rstrip("\n")
                if re.match(r'^\d+\t', line):  # line starts with a number followed by tab
                    if buffer:
                        outfile.write(buffer + "\n")
                    buffer = line
                else:
                    buffer += " " + line.strip()

            if buffer:
                outfile.write(buffer + "\n")

    counter+=1
print("All files cleaned and saved to:", output_dir)


1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250 251 252 253 254 255 256 257 258 259 260 261 262 263 264 265 266 267 268 269 270 271 272 273 274 275 276 277 

In [6]:
input_path = "/mnt/isilon/wang_lab/shared/pubmed_central_decompressed/batch_googleBERT_cased_15k_LLM_label_only_filtered_input/1-PMC10000012-PMC10024304.tsv"
output_path = "/mnt/isilon/wang_lab/shared/pubmed_central_decompressed/batch_googleBERT_cased_15k_LLM_label_only_filtered_input/1-PMC10000012-PMC10024304.clean.tsv"

with open(input_path, "r", encoding="utf-8") as infile, open(output_path, "w", encoding="utf-8") as outfile:
    buffer = ""
    for line in infile:
        # Strip newline characters
        line = line.rstrip("\n")
        if line.startswith("PMC"):
            if buffer:
                outfile.write(buffer + "\n")  # write completed previous line
            buffer = line  # start new line
        else:
            buffer += " " + line.strip()  # merge continuation line with space
    if buffer:
        outfile.write(buffer + "\n")  # write the last buffer


In [8]:
import pandas as pd
import os
batch_pmc_path='/mnt/isilon/wang_lab/shared/pubmed_central_decompressed/batch_BioMedBERT_15k_LLM_only_label_only_filtered_input'
filename='1-PMC10000012-PMC10024304.tsv'
file_path = os.path.join(batch_pmc_path, filename)
print(f"Loading: {filename}")

batch=pd.read_csv(file_path,sep='\t',lineterminator='\n')

Loading: 1-PMC10000012-PMC10024304.tsv


In [12]:
batch.loc[0].values

array(['PMC10000012', '1/12', 'abstract',
       'As the world braces to enter its fourth year of the coronavirus\ndisease 2019 (COVID-19) pandemic, the need for accessible and effective\nantiviral therapeutics continues to be felt globally. The recent surge\nof Omicron variant cases has demonstrated that vaccination and prevention\nalone cannot quell the spread of highly transmissible variants. A\nsafe and nontoxic therapeutic with an adaptable design to respond\nto the emergence of new variants is critical for transitioning to\nthe treatment of COVID-19 as an endemic disease. Here, we present\na novel compound, called SBCoV202, that specifically and tightly binds\nthe translation initiation site of RNA-dependent RNA polymerase within\nthe severe acute respiratory syndrome coronavirus 2 (SARS-CoV-2) genome,\ninhibiting viral replication. SBCoV202 is a Nanoligomer, a molecule\nthat includes peptide nucleic acid sequences capable of binding viral\nRNA with single-base-pair specificity t

In [15]:
import pandas as pd
import os
batch_pmc_path='/mnt/isilon/wang_lab/shared/pubmed_central_decompressed/batch_BioMedBERT_15k_LLM_only_label_only_filtered_input'
filename='1-PMC10000012-PMC10024304.clean.tsv'
file_path = os.path.join(batch_pmc_path, filename)
print(f"Loading: {filename}")

batch=pd.read_csv(file_path,sep='\t',lineterminator='\n')

Loading: 1-PMC10000012-PMC10024304.clean.tsv


In [16]:
batch.loc[0].values

array(['PMC10000012', '1/12', 'abstract',
       'As the world braces to enter its fourth year of the coronavirus disease 2019 (COVID-19) pandemic, the need for accessible and effective antiviral therapeutics continues to be felt globally. The recent surge of Omicron variant cases has demonstrated that vaccination and prevention alone cannot quell the spread of highly transmissible variants. A safe and nontoxic therapeutic with an adaptable design to respond to the emergence of new variants is critical for transitioning to the treatment of COVID-19 as an endemic disease. Here, we present a novel compound, called SBCoV202, that specifically and tightly binds the translation initiation site of RNA-dependent RNA polymerase within the severe acute respiratory syndrome coronavirus 2 (SARS-CoV-2) genome, inhibiting viral replication. SBCoV202 is a Nanoligomer, a molecule that includes peptide nucleic acid sequences capable of binding viral RNA with single-base-pair specificity to accurately 